# 1. Verificação de ruídos

In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# Crie ou carregue seu dataframe
df = pd.read_csv('abalone_dataset.csv')


def diagnosticar_ruidos(df):
    print("=== 1. VERIFICAÇÃO DE VALORES AUSENTES E SENTINELAS ===")
    print(df.isnull().sum())

    print("\n=== 2. RESUMO ESTATÍSTICO (Busque por Min/Max estranhos) ===")
    print(df.describe().T[["min", "mean", "50%", "max", "std"]])

    print("\n=== 3. OUTLIERS VIA IQR (POR COLUNA NUMÉRICA) ===")
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[
            (df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))
        ]
        pct = (len(outliers) / len(df)) * 100
        print(f"Coluna '{col}': {len(outliers)} outliers encontrados ({pct:.2f}%)")

    print("\n=== 4. ANOMALIAS MULTIVARIADAS (ISOLATION FOREST) ===")
    if len(num_cols) > 1:
        df_clean = df[num_cols].fillna(df[num_cols].median())
        iso = IsolationForest(contamination=0.05, random_state=42)
        preds = iso.fit_predict(df_clean)
        num_anomalias = (preds == -1).sum()
        print(
            f"Anomalias/ruídos combinados detectados: {num_anomalias} de {len(df)} linhas ({(num_anomalias/len(df))*100:.2f}%)"
        )


# Executar no seu dataframe
diagnosticar_ruidos(df)

=== 1. VERIFICAÇÃO DE VALORES AUSENTES E SENTINELAS ===
sex               0
length            0
diameter          0
height            0
whole_weight      0
shucked_weight    0
viscera_weight    0
shell_weight      0
type              0
dtype: int64

=== 2. RESUMO ESTATÍSTICO (Busque por Min/Max estranhos) ===
                   min      mean     50%     max       std
length          0.0750  0.521392  0.5400  0.8150  0.120756
diameter        0.0550  0.405865  0.4200  0.6500  0.099600
height          0.0000  0.138263  0.1400  0.5150  0.039206
whole_weight    0.0020  0.818738  0.7870  2.8255  0.489560
shucked_weight  0.0010  0.355398  0.3305  1.4880  0.221473
viscera_weight  0.0005  0.178349  0.1680  0.7600  0.109554
shell_weight    0.0015  0.235616  0.2250  1.0050  0.139215
type            1.0000  1.991379  2.0000  3.0000  0.824561

=== 3. OUTLIERS VIA IQR (POR COLUNA NUMÉRICA) ===
Coluna 'length': 47 outliers encontrados (1.50%)
Coluna 'diameter': 36 outliers encontrados (1.15%)
Coluna 

# 2. Verificação de linhas com o peso total menor que a soma dos pesos

In [ ]:
import pandas as pd

# 1. Carregar o dataset
df = pd.read_csv("abalone_dataset.csv")

# 2. Calcular a soma dos três sub-pesos
soma_sub_pesos = (
    df["shucked_weight"] + df["viscera_weight"] + df["shell_weight"]
)

# 3. Filtrar as linhas onde a soma é maior que o whole_weight
condicao = soma_sub_pesos != df["whole_weight"]
linhas_inconsistentes = df[condicao]

# 4. Resultados
total_linhas = len(linhas_inconsistentes)
porcentagem = (total_linhas / len(df)) * 100

print(f"Total de linhas onde a soma é maior: {total_linhas}")
print(f"Porcentagem do dataset: {porcentagem:.2f}%")

# Exibir as primeiras linhas inconsistentes
print("\nPrimeiros registros onde a soma ultrapassa o peso total:")
print(
    linhas_inconsistentes[
        ["whole_weight", "shucked_weight", "viscera_weight", "shell_weight"]
    ].head()
)

Total de linhas onde a soma é maior: 117
Porcentagem do dataset: 3.74%

Primeiros registros onde a soma ultrapassa o peso total:
     whole_weight  shucked_weight  viscera_weight  shell_weight
2          0.0290          0.0120          0.0075        0.0095
4          0.5765          0.3595          0.1350        0.2250
7          1.0125          0.5325          0.4365        0.3240
12         0.5615          0.2520          0.1370        0.1800
125        0.4335          0.2105          0.1205        0.1200


# 3. Verificação de linhas repetidas

In [2]:
import pandas as pd

CAMINHO = "abalone_dataset.csv"  # troque pelo caminho do seu arquivo

df = pd.read_csv(CAMINHO)

# Número da linha no arquivo CSV (índice 0 = linha 2, pois a linha 1 é o cabeçalho)
df["linha_no_arquivo"] = df.index + 2

colunas = [c for c in df.columns if c != "linha_no_arquivo"]

# keep=False marca TODAS as ocorrências (a original e as cópias)
repetidas = df[df.duplicated(subset=colunas, keep=False)].sort_values(colunas)

print("Linhas envolvidas em repetições:", len(repetidas))
print("Cópias extras (além da 1ª ocorrência):", df.duplicated(subset=colunas).sum())
repetidas

Linhas envolvidas em repetições: 0
Cópias extras (além da 1ª ocorrência): 0


,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,type,linha_no_arquivo


In [4]:
import pandas as pd

CAMINHO = "abalone_dataset.csv"  # troque pelo caminho do seu arquivo

df = pd.read_csv(CAMINHO)

df["linha_no_arquivo"] = df.index + 2   # número da linha no CSV

df.isna().sum()                                  # nulos por coluna
(df.select_dtypes("number") <= 0).sum()          # zeros e negativos
df.duplicated().sum()                            # linhas repetidas
df["sex"].value_counts()                         # categorias inconsistentes
df.describe()                                    # mínimos e máximos suspeitos
df.dtypes                                        # tipos errados

sex                     str
length              float64
diameter            float64
height              float64
whole_weight        float64
shucked_weight      float64
viscera_weight      float64
shell_weight        float64
type                  int64
linha_no_arquivo      int64
dtype: object